# 02_dim_tracks

DML: dim_tracks — Track dimension, PK: track_id.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

src = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_tracks")
    .filter(F.col("ingestion_date") == ingestion_date)
    .filter(F.col("run_id") == run_id)
    .select(
        "track_id", "track_name", "duration_ms", "popularity", "explicit",
        "album_id", "album_name", "album_release_date", "ingestion_date",
    )
    .withColumn("duration_min", F.round(F.col("duration_ms") / 60000.0, 4))
    .withColumn(
        "album_release_date",
        F.coalesce(
            F.to_date("album_release_date", "yyyy-MM-dd"),
            F.to_date("album_release_date", "yyyy-MM"),
            F.to_date("album_release_date", "yyyy"),
        ),
    )
    .withColumn("_valid_from", F.col("ingestion_date"))
    .drop("ingestion_date")
    .dropDuplicates(["track_id"])
)

upsert_delta(src, f"{CATALOG}.{SILVER_SCHEMA}.dim_tracks", ["track_id"])
print(f"dim_tracks: {src.count()} rows upserted")